# Example 2: Archival Ingest

Download, query, and explore AIS data.

This notebook shows the core archival workflow: download data from a source,
query it with Polars or SQL, and use helpers for common operations.

## Prerequisites

```bash
pip install neptune-ais[sql,notebooks]
```

**Data requirement:** This example downloads data from NOAA, which requires network access.
The download step fetches ~50–200 MB per day depending on the date. If you already have
local data, skip the download cell and point `cache_dir` to your existing store.

## Imports

In [1]:
from neptune_ais.api import Neptune
from neptune_ais.helpers import latest_positions, snapshot, vessel_history

## Download a Day of NOAA Data

The `Neptune` constructor specifies the date range and source(s).
Calling `.download()` triggers the full pipeline:

**fetch** → **normalize** → **QC** → **partition** → **catalog**

In [2]:
n = Neptune("2024-06-15", sources=["noaa"], cache_dir="/tmp/neptune_demo")
n.download()

['positions/noaa/2024-06-15']

## Query Positions as a Polars LazyFrame

`.positions()` returns a Polars `LazyFrame` — queries are not executed until you call
`.collect()`. This enables predicate pushdown and projection pruning for efficient queries
over large datasets.

In [3]:
positions = n.positions()
print(f"Positions: {positions.collect().shape}")

Positions: (9985832, 20)


## Latest Known Position per Vessel

`latest_positions()` returns the most recent position report for each unique MMSI.
Useful for fleet snapshots and "where are they now?" queries.

In [4]:
latest = latest_positions(positions)
print(f"Unique vessels: {latest.collect().shape[0]}")

Unique vessels: 23403


## Point-in-Time Snapshot

`snapshot()` finds the closest position report to a given timestamp for each vessel.
This is useful for reconstructing the state of the maritime domain at a specific moment.

In [5]:
snap = snapshot(positions, when="2024-06-15T12:00:00")
print(f"Vessels at noon: {snap.collect().shape[0]}")

Vessels at noon: 23403


## SQL Queries via DuckDB

`.sql()` runs SQL queries directly over your cataloged data using DuckDB.
The `positions` table is automatically registered.

In [6]:
result = n.sql(
    "SELECT mmsi, count(*) as n FROM positions GROUP BY mmsi ORDER BY n DESC LIMIT 10"
)
print(result)

┌───────────┬───────┐
│   mmsi    │   n   │
│   int64   │ int64 │
├───────────┼───────┤
│ 367161960 │  1388 │
│ 368191850 │  1386 │
│ 367616260 │  1386 │
│ 367664560 │  1383 │
│ 368091590 │  1375 │
│ 366935020 │  1375 │
│ 367002000 │  1375 │
│ 367458840 │  1374 │
│ 368311450 │  1373 │
│ 368179250 │  1368 │
└───────────┴───────┘
       10 rows     



## Vessel History

`vessel_history()` retrieves the full history for a single vessel across all
datasets — positions, tracks, and events — returned as a dictionary of LazyFrames.

In [8]:
history = vessel_history(367161960, positions=positions)
# Returns dict[str, LazyFrame] with keys like "positions", "tracks", "events".
print(f"Positions for vessel: {history['positions'].collect().shape[0]}")

Positions for vessel: 1388


## Next Steps

- **[03 — Multi-Source Fusion](03_multi_source_fusion.ipynb)**: Combine data from multiple providers
- **[04 — Event Detection](04_event_detection.ipynb)**: Derive port calls, encounters, and more